# 03 Cohort Construction

This notebook builds the adult ICU cohort and implements a reproducible Sepsis-3-aligned labeling pipeline using suspected infection and SOFA score increase. It saves intermediate artifacts so the cohort definition can be audited and reused by downstream notebooks.

## Important methodological note

The default implementation below is **Sepsis-3 aligned and leakage-safe**, but its exact SOFA variable extraction depends on local `D_ITEMS.csv` and `D_LABITEMS.csv` label matching. Before paper submission, the resolved item sets should be clinically validated against the local MIMIC dictionary tables and refined if needed.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy

In [3]:
import pandas as pd

from src.utils.config import load_config
from src.utils.paths import ensure_directories, resolve_project_paths
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.data_processing.io import list_available_tables, validate_required_tables
from src.data_processing.cohort import build_base_icu_cohort, summarize_cohort, create_patient_level_splits
from src.data_processing.sepsis3 import (
    compute_sofa_hourly,
    derive_sepsis_labels,
    derive_suspected_infection,
    detect_antibiotic_orders,
    detect_culture_orders,
    extract_sofa_measurements,
)

In [4]:

available_tables = list_available_tables(paths['extracted_data_dir'])
required_status = validate_required_tables(paths['extracted_data_dir'], config['dataset']['required_tables'])
assert all(required_status.values()), 'Some required tables are missing. Run 01_dataset_setup first.'

available_tables[:20]

['ADMISSIONS.csv',
 'CALLOUT.csv',
 'CAREGIVERS.csv',
 'CHARTEVENTS.csv',
 'CPTEVENTS.csv',
 'DATETIMEEVENTS.csv',
 'DIAGNOSES_ICD.csv',
 'DRGCODES.csv',
 'D_CPT.csv',
 'D_ICD_DIAGNOSES.csv',
 'D_ICD_PROCEDURES.csv',
 'D_ITEMS.csv',
 'D_LABITEMS.csv',
 'ICUSTAYS.csv',
 'INPUTEVENTS_CV.csv',
 'INPUTEVENTS_MV.csv',
 'LABEVENTS.csv',
 'MICROBIOLOGYEVENTS.csv',
 'NOTEEVENTS.csv',
 'OUTPUTEVENTS.csv']

## Build the base ICU cohort

In [5]:
cohort_df = build_base_icu_cohort(
    extracted_dir=paths['extracted_data_dir'],
    adult_age_min=config['cohort']['adult_age_min'],
    min_icu_los_hours=config['cohort']['min_icu_los_hours'],
    first_icu_only=config['cohort']['first_icu_only'],
    low_memory=config['dataset']['low_memory'],
)
cohort_summary = summarize_cohort(cohort_df)
cohort_summary

{'icu_stay_count': 52904,
 'patient_count': 38312,
 'admission_count': 49432,
 'median_icu_los_hours': 51.83305555555556,
 'median_age_years': 64.0,
 'masked_age_89_plus_count': 2689}

In [6]:
cohort_df.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,FIRST_CAREUNIT,LAST_CAREUNIT,INTIME,OUTTIME,ADMITTIME,DISCHTIME,DEATHTIME,ETHNICITY,GENDER,DOB,DOD,icu_los_hours,age_at_icu_intime_raw,age_is_masked_89_plus,age_at_icu_intime,is_adult_icu
0,3,145834,211552,MICU,MICU,2101-10-20 19:10:11,2101-10-26 20:43:09,2101-10-20 19:08:00,2101-10-31 13:58:00,NaT,WHITE,M,2025-04-11,2102-06-14,145.549444,76.0,False,76.0,True
1,4,185777,294638,MICU,MICU,2191-03-16 00:29:31,2191-03-17 16:46:31,2191-03-16 00:28:00,2191-03-23 18:41:00,NaT,WHITE,F,2143-05-12,NaT,40.283333,47.0,False,47.0,True
2,6,107064,228232,SICU,SICU,2175-05-30 21:30:54,2175-06-03 13:39:54,2175-05-30 07:15:00,2175-06-15 16:00:00,NaT,WHITE,F,2109-06-21,NaT,88.150000,65.0,False,65.0,True
3,9,150750,220597,MICU,MICU,2149-11-09 13:07:02,2149-11-14 20:52:14,2149-11-09 13:06:00,2149-11-14 10:15:00,2149-11-14 10:15:00,UNKNOWN/NOT SPECIFIED,M,2108-01-26,2149-11-14,127.753333,41.0,False,41.0,True
4,11,194540,229441,SICU,SICU,2178-04-16 06:19:32,2178-04-17 20:21:05,2178-04-16 06:18:00,2178-05-11 19:00:00,NaT,WHITE,F,2128-02-22,2178-11-14,38.025833,50.0,False,50.0,True


## Detect suspected infection

This step implements the Sepsis-3 temporal pairing logic between antibiotic administration and culture sampling.

In [7]:
antibiotics_df = detect_antibiotic_orders(
    extracted_dir=paths['extracted_data_dir'],
    antibiotic_keywords=config['sepsis3']['antibiotic_keywords'],
    low_memory=config['dataset']['low_memory'],
)
cultures_df = detect_culture_orders(
    extracted_dir=paths['extracted_data_dir'],
    low_memory=config['dataset']['low_memory'],
)
suspected_infection_df = derive_suspected_infection(
    antibiotics=antibiotics_df,
    cultures=cultures_df,
    culture_after_antibiotic_hours=config['sepsis3']['suspicion_window']['culture_after_antibiotic_hours'],
    antibiotic_after_culture_hours=config['sepsis3']['suspicion_window']['antibiotic_after_culture_hours'],
)
suspected_infection_df.head()

,SUBJECT_ID,HADM_ID,suspected_infection_time,culture_time,antibiotic_time
0,54610,100003,2150-04-17 00:00:00,2150-04-17 18:41:00,2150-04-17
1,9895,100006,2108-04-10 12:00:00,2108-04-10 12:00:00,2108-04-12
2,23018,100007,2145-03-31 00:25:00,2145-03-31 00:25:00,2145-04-02
3,533,100009,2162-05-16 16:47:00,2162-05-16 16:47:00,2162-05-17
4,87977,100011,2177-08-29 00:00:00,2177-08-29 06:08:00,2177-08-29


## Resolve SOFA variables and compute hourly proxy SOFA

SOFA variables are resolved from dictionary tables when available. The resulting hourly table is the basis for sepsis onset labeling and is saved for auditability.

In [8]:
measurements = extract_sofa_measurements(
    extracted_dir=paths['extracted_data_dir'],
    cohort=cohort_df,
    lab_item_keywords=config['sepsis3']['lab_item_keywords'],
    chart_item_keywords=config['sepsis3']['chart_item_keywords'],
    vasopressor_keywords=config['sepsis3']['vasopressor_keywords'],
    chunksize=config['dataset']['chunksize'],
    low_memory=config['dataset']['low_memory'],
)
sofa_hourly_df = compute_sofa_hourly(measurements, cohort_df)
sofa_hourly_df.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,hour,respiratory,coagulation,liver,cardiovascular,cns,renal,total_sofa
0,2,163353.0,243653.0,2138-07-17 20:00:00,0,0,0,1,0,0,1
1,3,145834.0,211552.0,2101-10-20 18:00:00,0,0,0,1,0,0,1
2,3,145834.0,211552.0,2101-10-20 19:00:00,0,0,0,1,0,2,3
3,3,145834.0,211552.0,2101-10-20 20:00:00,1,0,0,1,0,0,2
4,3,145834.0,211552.0,2101-10-20 21:00:00,0,0,0,1,0,0,1


In [9]:
measurements['lab_itemids'].head(), measurements['chart_itemids'].head()

(     concept  itemid
 0   platelet   51240
 1   platelet   51264
 2   platelet   51265
 3   platelet   51266
 4  bilirubin   50838,
   concept  itemid
 0     map      52
 1     map     443
 2     map     456
 3     map    2294
 4     map    2647)

## Derive sepsis onset labels

In [10]:
labels_df = derive_sepsis_labels(
    cohort=cohort_df,
    suspected_infection=suspected_infection_df,
    sofa_hourly=sofa_hourly_df,
    baseline_window_hours=config['sepsis3']['baseline_window_hours'],
    sofa_delta_threshold=config['sepsis3']['sofa_delta_threshold'],
    sofa_before_suspicion_hours=config['sepsis3']['onset_window']['sofa_before_suspicion_hours'],
    sofa_after_suspicion_hours=config['sepsis3']['onset_window']['sofa_after_suspicion_hours'],
)
labels_df.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,suspected_infection_time,baseline_sofa,max_sofa_delta,sepsis_onset_time,sepsis3_label
0,3,145834,211552,NaT,0.0,6.0,NaT,0
1,4,185777,294638,2191-03-16,0.0,1.0,NaT,0
2,6,107064,228232,NaT,0.0,4.0,NaT,0
3,9,150750,220597,2149-11-10,0.0,7.0,2149-11-09 13:00:00,1
4,11,194540,229441,NaT,0.0,2.0,NaT,0


In [11]:
label_summary = {
    'cohort_stays': int(cohort_df['ICUSTAY_ID'].nunique()),
    'suspected_infection_admissions': int(suspected_infection_df['HADM_ID'].nunique()) if not suspected_infection_df.empty else 0,
    'sepsis_positive_stays': int(labels_df.loc[labels_df['sepsis3_label'] == 1, 'ICUSTAY_ID'].nunique()),
    'positive_rate': float(labels_df['sepsis3_label'].mean()) if not labels_df.empty else 0.0,
}
label_summary

{'cohort_stays': 52904,
 'suspected_infection_admissions': 32694,
 'sepsis_positive_stays': 15418,
 'positive_rate': 0.29143353999697563}

## Create patient-level splits

In [12]:
splits_df = create_patient_level_splits(
    cohort=cohort_df,
    val_size=config['training']['val_size'],
    test_size=config['training']['test_size'],
    random_state=config['cohort']['patient_split_random_state'],
)
splits_df['split'].value_counts()

split
train    36946
test      8009
val       7949
Name: count, dtype: int64

## Save reproducible artifacts

In [13]:
output_dir = paths['processed_data_dir'] / '03_cohort_construction'
artifact_bundle = {
    'cohort': cohort_df,
    'antibiotics': antibiotics_df,
    'cultures': cultures_df,
    'suspected_infection': suspected_infection_df,
    'sofa_hourly': sofa_hourly_df,
    'sepsis_labels': labels_df,
    'patient_splits': splits_df,
    'resolved_lab_itemids': measurements['lab_itemids'],
    'resolved_chart_itemids': measurements['chart_itemids'],
}
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'cohort': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/cohort.csv',
 'antibiotics': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/antibiotics.csv',
 'cultures': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/cultures.csv',
 'suspected_infection': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/suspected_infection.csv',
 'sofa_hourly': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/sofa_hourly.csv',
 'sepsis_labels': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/sepsis_labels.csv',
 'patient_splits': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/03_cohort_construction/patient_splits.csv',
 'resolved_lab_itemids': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/

In [14]:
manifest_path = paths['manifests_dir'] / '03_cohort_construction_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='03_cohort_construction',
    config=config,
    extra={
        'cohort_summary': cohort_summary,
        'label_summary': label_summary,
        'saved_artifacts': saved_paths,
        'dictionary_tables_available': {
            'D_ITEMS.csv': 'D_ITEMS.csv' in available_tables,
            'D_LABITEMS.csv': 'D_LABITEMS.csv' in available_tables,
        },
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/manifests/03_cohort_construction_manifest.json')

## Interpretation checklist

Before moving to feature engineering, review:

- How many ICU stays remain after the adult and ICU length filters?
- How often are suspected infection events identified?
- Which SOFA concepts resolved to item IDs successfully?
- Is the positive rate clinically plausible?
- Do any labeling assumptions need refinement before model training?

## Next notebook

`04_feature_engineering.ipynb` will transform this cohort into leakage-safe hourly structured trajectories for 6h, 12h, and 24h early prediction.